# R1-02 Data prep
Builds `data/processed/*` and `data/README.md` from the two raw files in `data/raw/` (git-ignored).
- Shopzilla (Kaggle): https://www.kaggle.com/datasets/ddosad/ecommerce-customer-service-satisfaction
- Tobi-Bueck (Hugging Face): https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets

**Deviation from the spec (Ayhan's instruction):** the spec says Shopzilla is filtered to the email channel. We keep ALL channels in `shopzilla_contacts_clean.csv` and also save an email-only file `shopzilla_email_clean.csv`. Verify checks the email file has only Email rows.

**Pseudonymised teams:** each Supervisor becomes `Team NN`. The numbering comes from an UNSEEDED random shuffle, so **labels change on every rerun** and cannot be rebuilt from the public data. The mapping and all name lists are deleted and never saved.

**Minimum group:** a team-month needs at least 5 DISTINCT agents, otherwise its rows go to `Other`. If `Other` itself has fewer than 5 distinct agents in a month, those rows are dropped (the count is printed).

**E-commerce filter (R1-02 fix):** after the queue filter, Tobi-Bueck tickets are kept only if subject OR body contains a shop keyword at the start of a word (list `SHOP_KEYWORDS` in cell 3b; "orders" and "returned" match, so does "charger"). The idioms "in order to", "in charge" and "in return" are ignored for matching only. If fewer than 200 tickets remain, the notebook stops with a WARNING before sampling or writing (top up with synthetic emails).

**Stratified sample:** test 50 = 10 random tickets per queue (4 queues) + 10 more from the remaining pool; tuning 150 = random from what is left (seed 42, disjoint). If a queue has fewer than 10 tickets the notebook stops (top up with synthetic emails).

**Labelling sheet:** cell 3e rewrites `langsmith/heldout_priority_labels.csv` (50 held-out emails, blank labels). An existing sheet is kept only if it has the same 50 ids and labels already filled; otherwise the old sheet is copied to the backup folder first.

Run top to bottom (Restart & Run All). Each step prints PASS or FAIL. Undo only runs if `RUN_UNDO = True`.

In [ ]:
# 0 Settings
from pathlib import Path
import subprocess, shutil, datetime, hashlib, re, random
import pandas as pd, numpy as np

_top = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True).stdout.strip()
REPO = Path(_top) if _top else Path.cwd()
RAW = REPO / 'data' / 'raw'
PROC = REPO / 'data' / 'processed'
SHOP_FILE = RAW / 'shopzilla_cs_satisfaction.csv'
TICKETS_FILE = RAW / 'tobi_bueck_tickets.csv'
NOTEBOOK = REPO / 'notebooks' / 'R1-02_data_prep.ipynb'
# every file this notebook creates or changes
OUT = {
    'contacts': PROC / 'shopzilla_contacts_clean.csv',
    'email': PROC / 'shopzilla_email_clean.csv',
    'tuning': PROC / 'tobi_tuning_150.csv',
    'test': PROC / 'tobi_test_50.csv',
    'pulse': PROC / 'synthetic_team_pulse.csv',
    'pulse_summary': PROC / 'synthetic_team_pulse_summary.csv',
    'readme': REPO / 'data' / 'README.md',
    'labels': REPO / 'langsmith' / 'heldout_priority_labels.csv',   # labelling sheet for the 50 held-out emails
}
RUN_UNDO = False
MIN_GROUP = 5
results = []

# patterns for personal data in free text
EMAIL_RE = r'[\w.+-]+@[\w-]+(?:\.[\w-]+)+'
URL_RE = r'(?:https?://|www\.)\S+'
PHONE_RE = r'\+?\d[\d\s().-]{7,}\d'

def check(step, ok, note=''):
    results.append((step, 'PASS' if ok else 'FAIL', note))
    print('PASS' if ok else 'FAIL', '|', step, '|', note)

# common English words that must not count as name tokens
STOP = set('young long strong page price will grace hope may mark rose joy king white black brown green hall wood bell hill lane love rich west park best more case call order other team time good great day june april august summer hunter cook baker miller smith fisher chase dean faith gay glen guy ken lee ray art bill bob cash crystal daisy dawn honey iris ivy jack jade lily max pat penny ruby sky star sue tom victor'.split())

def name_tokens(df):
    # first/last name tokens (length >= 3, lower case, not in STOP) from raw Agent_name, Supervisor, Manager
    toks = set()
    for col in ['Agent_name', 'Supervisor', 'Manager']:
        for full in df[col].dropna().unique():
            toks |= {t.lower() for t in re.findall(r'[^\W\d_]+', str(full)) if len(t) >= 3}
    return toks - STOP

def snapshot():
    # sha256 of an explicit list of files we do not own (never .env, never logs/board)
    files = list(RAW.glob('*.csv')) + [REPO / 'AGENTS.md', REPO / 'CLAUDE.md', REPO / '.gitignore']
    files += list((REPO / 'specs').glob('*.md'))
    files += [p for p in (REPO / 'notebooks').glob('*.ipynb') if p != NOTEBOOK]
    # R1-05 files next to the labelling sheet (the sheet itself is ours, so it is skipped)
    files += [p for p in (REPO / 'langsmith').glob('*') if p.is_file() and p != OUT['labels'] and not p.name.startswith('.env')]
    files += [PROC / 'synthetic_policy_cases.csv']
    return {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files if p.exists()}

BEFORE = snapshot()
print('Files hashed before:', len(BEFORE))

In [ ]:
# 1 Check: right folder, input files exist, raw facts match
check('repo root found', (REPO / 'AGENTS.md').exists(), str(REPO))
check('Shopzilla raw file exists', SHOP_FILE.exists(), SHOP_FILE.name)
check('Tobi-Bueck raw file exists', TICKETS_FILE.exists(), TICKETS_FILE.name)
check('data/raw is git-ignored', 'data/raw/' in (REPO / '.gitignore').read_text())
assert all(r[1] == 'PASS' for r in results), 'Stop: fix the FAIL above (run R1-02a first)'

shop_raw = pd.read_csv(SHOP_FILE)
tix_raw = pd.read_csv(TICKETS_FILE)   # default quoting keeps newlines inside bodies
check('Shopzilla has 85,907 rows', len(shop_raw) == 85907, str(len(shop_raw)))
check('Shopzilla has 3,023 Email rows', (shop_raw['channel_name'] == 'Email').sum() == 3023)
check('Shopzilla channel_name has no NaN', shop_raw['channel_name'].notna().all())
check('Tobi-Bueck has 61,765 rows', len(tix_raw) == 61765, str(len(tix_raw)))
assert all(r[1] == 'PASS' for r in results), 'Stop: raw files do not match the expected facts'

In [ ]:
# 2 Backup: copy every output file that already exists to _backup/<timestamp>/
# (OUT includes data/README.md and langsmith/heldout_priority_labels.csv)
TS = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
BK = REPO / '_backup' / TS
existed = {}
for key, p in OUT.items():
    existed[key] = p.exists()
    if p.exists():
        dest = BK / p.relative_to(REPO)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
print('Already existing (backed up):', [k for k, v in existed.items() if v] or 'none')
print('Backup folder:', BK)

In [ ]:
# 3a Create: Shopzilla (all channels + email-only file)
PROC.mkdir(parents=True, exist_ok=True)
shop = shop_raw.copy()

# build the name tokens once, then delete the raw data; only the token set stays until Verify
NAME_TOK = name_tokens(shop_raw)
del shop_raw

# timestamps are day-first, e.g. 01/08/2023 11:13; print how many failed to parse
for col in ['order_date_time', 'Issue_reported at', 'issue_responded']:
    shop[col] = pd.to_datetime(shop[col], format='%d/%m/%Y %H:%M', errors='coerce')
shop['Survey_response_Date'] = pd.to_datetime(shop['Survey_response_Date'], format='%d-%b-%y', errors='coerce')
print('NaT per date column:', shop[['order_date_time', 'Issue_reported at', 'issue_responded', 'Survey_response_Date']].isna().sum().to_dict())

# response time in hours; negative values are errors, set to NaN
shop['response_hours'] = (shop['issue_responded'] - shop['Issue_reported at']).dt.total_seconds() / 3600
print('response_hours NaN:', shop['response_hours'].isna().sum(), '| negative:', (shop['response_hours'] < 0).sum())
shop.loc[shop['response_hours'] < 0, 'response_hours'] = np.nan
shop['response_valid'] = shop['response_hours'].notna()

def scrub_names(s):
    return re.sub(r'[^\W\d_]+', lambda m: '[NAME]' if m.group().lower() in NAME_TOK else m.group(), s)

# scrub Customer Remarks: emails, urls, phones, then names (needed for UC2, so kept)
old = shop['Customer Remarks']
new = (old.str.replace(EMAIL_RE, '[EMAIL]', regex=True)
          .str.replace(URL_RE, '[URL]', regex=True)
          .str.replace(PHONE_RE, '[PHONE]', regex=True)
          .map(lambda s: scrub_names(s) if isinstance(s, str) else s))
print('Remarks changed by scrubbing:', int((old.notna() & (old != new)).sum()), 'of', int(old.notna().sum()))
shop['Customer Remarks'] = new

# Supervisor -> Team NN with an UNSEEDED random order (labels change on every rerun)
sups = sorted(shop['Supervisor'].dropna().unique())
random.SystemRandom().shuffle(sups)
team_map = {name: 'Team ' + str(i).zfill(2) for i, name in enumerate(sups, 1)}
shop['team'] = shop['Supervisor'].map(team_map).fillna('Other')
print('Supervisors mapped to teams:', len(sups))

def apply_min_group(df):
    # a team-month needs >= 5 DISTINCT agents, else 'Other'; Other with < 5 agents is dropped
    month = df['Issue_reported at'].dt.to_period('M')
    n = df['Agent_name'].groupby([df['team'], month]).transform('nunique').fillna(0)
    df['team'] = df['team'].where(n >= MIN_GROUP, 'Other')
    n = df['Agent_name'].groupby([df['team'], month]).transform('nunique').fillna(0)
    df['group_ok'] = n >= MIN_GROUP   # computed before Agent_name is dropped
    dropped = int((~df['group_ok']).sum())
    return df[df['group_ok']].copy(), dropped

# email-only file is masked on its own rows, so its groups are also >= 5 agents
email = shop[shop['channel_name'] == 'Email'].copy()
shop, dropped_all = apply_min_group(shop)
email, dropped_email = apply_min_group(email)
other_all = (shop['team'] == 'Other').mean()
other_email = (email['team'] == 'Other').mean()
print('Rows dropped (Other < 5 agents): all channels', dropped_all, '| email', dropped_email)
print('Other share: all channels', round(other_all, 3), '| email', round(other_email, 3))

# assert every kept (team, month) has >= 5 distinct agents, right before Agent_name is dropped
for df in (shop, email):
    m = df['Issue_reported at'].dt.to_period('M')
    assert (df['Agent_name'].groupby([df['team'], m]).nunique() >= MIN_GROUP).all(), 'a team-month has < 5 agents'

# drop everything that identifies a person, an order, or a small group
DROP = ['Agent_name', 'Supervisor', 'Manager', 'connected_handling_time', 'Order_id',
        'Unique id', 'Customer_City', 'Agent Shift', 'Tenure Bucket']
shop = shop.drop(columns=DROP)
email = email.drop(columns=DROP)
shop.to_csv(OUT['contacts'], index=False)
email.to_csv(OUT['email'], index=False)
del team_map, sups, old, new   # the team mapping and supervisor list are gone

# quick profile (medians use valid response times only)
print('\nDate range:', shop['Issue_reported at'].min(), 'to', shop['Issue_reported at'].max())
month = shop['Issue_reported at'].dt.to_period('M')
print('\nContacts per month per channel:\n', pd.crosstab(month, shop['channel_name']))
print('\nMean CSAT per channel:\n', shop.groupby('channel_name')['CSAT Score'].mean().round(2).to_string())
print('\nMedian response_hours per channel (valid rows only):\n', shop[shop['response_valid']].groupby('channel_name')['response_hours'].median().round(2).to_string())
print('\nEmail rows with Customer Remarks filled:', email['Customer Remarks'].notna().sum(), 'of', len(email))

In [ ]:
# 3b Create: Tobi-Bueck (latest version, e-commerce queues, e-commerce keywords, stratified 150 tuning + 50 held-out test)
tix = tix_raw.copy()
tix['version_num'] = pd.to_numeric(tix['version'], errors='coerce')   # blank -> NaN, excluded below
LATEST = tix['version_num'].max()
assert LATEST == 400, 'latest version is not 400: ' + str(LATEST)
print('Rows before:', len(tix), '| latest version:', LATEST)
tix = tix[tix['version_num'] == LATEST]
print('Rows after latest-version filter:', len(tix))

QUEUES = ['Customer Service', 'Returns and Exchanges', 'Billing and Payments', 'Product Support']
tix = tix[tix['queue'].isin(QUEUES) & tix['language'].isin(['en', 'de'])].drop(columns='version_num')
print('Rows after queue + language filter:', len(tix))

# remove duplicate tickets BEFORE sampling, so tuning and test cannot share a ticket
tix = tix.drop_duplicates(['subject', 'body'])
filtered = tix.copy()   # queue + language + dedup, before the keyword filter
print('Rows after dedup:', len(tix))

# e-commerce filter: keep a ticket only if its subject OR body matches a shop keyword
SHOP_KEYWORDS = ['order', 'delivery', 'shipping', 'parcel', 'return', 'refund', 'exchange', 'invoice',
                 'payment', 'charge', 'damaged', 'cancel',
                 'Bestellung', 'Lieferung', 'Paket', 'Versand', 'Rücksendung', 'Rückgabe', 'Erstattung',
                 'Umtausch', 'Rechnung', 'Zahlung', 'beschädigt', 'stornieren']
# English idioms that contain a keyword but are not about shop topics ('in order to', 'in charge of').
# They are removed from the text used for matching ONLY; the stored subject and body are never changed.
IDIOMS = ['in order to', 'in charge', 'in return']
# \s+ between the words, so extra spaces or a line break inside an idiom do not matter
IDIOM_PATTERN = r'\b(?:' + '|'.join(r'\s+'.join(map(re.escape, i.split())) for i in IDIOMS) + r')\b'
# a keyword must start at a word start, so 'orders' and 'returned' match but 'border' does not
# (prefix rule: 'charge' also matches 'charger' and 'charging')
KW_PATTERN = r'\b(?:' + '|'.join(map(re.escape, SHOP_KEYWORDS)) + ')'

def has_keyword(df):
    # fillna('') before astype(str): pandas 3 keeps NaN in astype(str), and subject is often blank
    subject = df['subject'].fillna('').astype(str).str.replace(IDIOM_PATTERN, ' ', case=False, regex=True)
    body = df['body'].fillna('').astype(str).str.replace(IDIOM_PATTERN, ' ', case=False, regex=True)
    return subject.str.contains(KW_PATTERN, case=False, regex=True) | body.str.contains(KW_PATTERN, case=False, regex=True)

tix = tix[has_keyword(tix)]
print('Rows after e-commerce keyword filter:', len(tix))
print('\nTickets per queue and language after the keyword filter:')
print(pd.crosstab(tix['queue'], tix['language'], margins=True))

# stop before sampling and writing if there are too few tickets (then top up with synthetic emails)
if len(tix) < 200:
    print('WARNING: only', len(tix), 'e-commerce tickets remain (need 200). Top up with synthetic emails. Stopping before sampling and writing.')
    raise RuntimeError('Stopped: fewer than 200 e-commerce tickets after the keyword filter (' + str(len(tix)) + ').')
queue_counts = tix['queue'].value_counts()
short = [q for q in QUEUES if queue_counts.get(q, 0) < 10]
if short:
    print('WARNING: fewer than 10 tickets in queue(s):', short, '. Top up with synthetic emails. Stopping before sampling and writing.')
    raise RuntimeError('Stopped: a queue has fewer than 10 tickets, top up with synthetic emails: ' + str(short))

# stratified sample (seed 42): test 50 = 10 per queue (4 x 10 = 40) + 10 from the remaining pool
test_core = pd.concat([tix[tix['queue'] == q].sample(n=10, random_state=42) for q in QUEUES])
pool = tix.drop(test_core.index)
test_extra = pool.sample(n=10, random_state=42)
test_df = pd.concat([test_core, test_extra]).sample(frac=1, random_state=42)   # shuffle
# tuning 150 = random from what is left after the test set, so the two sets cannot overlap
tune_df = pool.drop(test_extra.index).sample(n=150, random_state=42)

# tuning first (T001-T150), then test (T151-T200)
sample = pd.concat([tune_df, test_df]).reset_index(drop=True)
sample.insert(0, 'ticket_id', ['T' + str(i).zfill(3) for i in range(1, 201)])
sample['split'] = np.where(sample.index < 150, 'tuning', 'test')

# 10 of the 50 test rows are used for the timing test
sample['timing_test'] = False
timing_idx = sample[sample['split'] == 'test'].sample(n=10, random_state=42).index
sample.loc[timing_idx, 'timing_test'] = True

# PII scan on the sample: count, then mask emails and phones
TEXT_COLS = ['subject', 'body', 'answer']
def pii_count(df):
    return int(sum(df[c].str.count(EMAIL_RE).sum() + df[c].str.count(PHONE_RE).sum() for c in TEXT_COLS))
print('PII hits before masking:', pii_count(sample))
for col in TEXT_COLS:
    sample[col] = sample[col].str.replace(EMAIL_RE, '[EMAIL]', regex=True).str.replace(PHONE_RE, '[PHONE]', regex=True)
print('PII hits after masking:', pii_count(sample))

sample[sample['split'] == 'tuning'].to_csv(OUT['tuning'], index=False)
sample[sample['split'] == 'test'].to_csv(OUT['test'], index=False)
print(sample['split'].value_counts().to_string())
for name in ['tuning', 'test']:
    print('\nQueue x language,', name, ':')
    print(pd.crosstab(sample[sample['split'] == name]['queue'], sample[sample['split'] == name]['language'], margins=True))

In [ ]:
# 3c Create: synthetic team pulse (80 made-up answers, team level only)
rng = np.random.default_rng(42)
teams = ['Pulse team ' + ch for ch in 'ABCDEFGH']   # neutral labels, not linked to Shopzilla teams

texts = {
    'saves time': ['The draft gets me started fast.', 'I answer more emails per hour.'],
    'better quality': ['Replies sound clearer and more polite.', 'Fewer mistakes in my answers.'],
    "don't trust AI": ['I cannot see why it wrote this.', 'I check every line, so no time saved.'],
    'fear for my job': ['I worry AI will replace our team.', 'Not sure what this means for my role.'],
    'need training': ['I need a short training first.', 'Please show us how to use it.'],
    'other': ['Depends on the type of ticket.', 'No strong opinion yet.'],
}
# the category weights below are placeholder assumptions, not data
why = rng.choice(list(texts), size=80, p=[0.2, 0.15, 0.2, 0.1, 0.2, 0.15])
pulse = pd.DataFrame({
    'team': np.repeat(teams, 10),   # 10 answers per team, so every team has >= 5
    'uses_ai_answer': rng.choice(['Yes', 'No'], size=80),
    'why_category': why,
})
pulse['free_text'] = [texts[w][rng.integers(2)] for w in pulse['why_category']]
pulse['synthetic'] = True
pulse = pulse.sample(frac=1, random_state=42).reset_index(drop=True)   # shuffle, no respondent id
pulse.to_csv(OUT['pulse'], index=False)

# summary: counts per team x answer x reason; counts under 5 are suppressed as '<5'
summary = pulse.groupby(['team', 'uses_ai_answer', 'why_category']).size().reset_index(name='count')
summary['count'] = summary['count'].map(lambda n: '<5' if n < MIN_GROUP else str(n))
summary['synthetic'] = True
summary.to_csv(OUT['pulse_summary'], index=False)
print(pulse.groupby('team').size().to_string())

In [ ]:
# 3d Create: data/README.md (row counts come from the frames that were written)
n_contacts, n_email = len(shop), len(email)
n_tune, n_test = int((sample['split'] == 'tuning').sum()), int((sample['split'] == 'test').sum())
n_timing = int(sample['timing_test'].sum())
n_pulse, n_summary = len(pulse), len(summary)
KW_TEXT = ', '.join(SHOP_KEYWORDS)
IDIOM_TEXT = ', '.join('"' + i + '"' for i in IDIOMS)

# markdown table: held-out test set, queue x language (computed from the written test rows)
ct = pd.crosstab(sample[sample['split'] == 'test']['queue'], sample[sample['split'] == 'test']['language']).reindex(QUEUES).fillna(0).astype(int)
TEST_TABLE = '\n'.join(['| Queue | ' + ' | '.join(ct.columns) + ' |', '|---|' + '---|' * len(ct.columns)]
                       + ['| ' + q + ' | ' + ' | '.join(str(v) for v in ct.loc[q]) + ' |' for q in ct.index])

readme = f'''# Data

## Sources
- Shopzilla customer service contacts (Kaggle, eCommerce Customer Service Satisfaction): https://www.kaggle.com/datasets/ddosad/ecommerce-customer-service-satisfaction . Licence: to confirm (Kaggle page did not render).
- Tobi-Bueck customer support tickets (Hugging Face): https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets . Licence: CC BY-NC 4.0 (per Hugging Face licence tag, checked 2026-11-26).

## Raw files are not committed
`data/raw/` is git-ignored. Download it with notebook R1-02a. Raw files: `shopzilla_cs_satisfaction.csv`, `tobi_bueck_tickets.csv`.

## Processed files (`data/processed/`, made by notebook R1-02)
| File | Rows | What it is |
|---|---|---|
| shopzilla_contacts_clean.csv | {n_contacts} | All channels (Inbound, Outcall, Email), pseudonymised |
| shopzilla_email_clean.csv | {n_email} | Email channel only, pseudonymised |
| tobi_tuning_150.csv | {n_tune} | Tickets for tuning and demo (split = tuning) |
| tobi_test_50.csv | {n_test} | Held-out test tickets (split = test); {n_timing} have timing_test = True |
| synthetic_team_pulse.csv | {n_pulse} | Synthetic agent pulse, team level (synthetic = True) |
| synthetic_team_pulse_summary.csv | {n_summary} | Counts per team, answer and reason; counts under 5 shown as <5 (synthetic = True) |

## Column changes (Shopzilla)
- Added `response_hours` = issue_responded minus Issue_reported at, in hours. Negative values are set to empty. `response_valid` is True when response_hours is usable. Use valid rows only for medians.
- Added `team` = Team NN from the Supervisor, pseudonymised. The numbering is random on every run of the notebook, so labels differ between runs and the mapping is not saved.
- Added `group_ok` = True when the team-month has at least {MIN_GROUP} distinct agents (checked before agent names were dropped).
- Dropped `Agent_name`, `Supervisor`, `Manager`, `connected_handling_time`, `Order_id`, `Unique id`, `Customer_City`, `Agent Shift`, `Tenure Bucket`. The last four are a row-level re-identification risk. They can be added later only as aggregates with at least {MIN_GROUP} per group.
- Customer Remarks is kept for the root cause report (UC2). Emails, phone numbers and urls are replaced by [EMAIL], [PHONE], [URL]. Agent, supervisor and manager name parts are replaced by [NAME]. Other personal details in free text are possible (to confirm).
- Timestamps parsed day-first (dd/mm/yyyy hh:mm); Survey_response_Date parsed as dd-Mon-yy.

## Column changes (Tobi-Bueck)
- Kept only the latest version (400; blank versions excluded), duplicates on subject and body removed before sampling.
- Kept queues: Customer Service, Returns and Exchanges, Billing and Payments, Product Support. Languages: en, de.
- E-commerce keyword filter (applied after the queue filter and dedup): a ticket is kept only if its subject OR its body contains one of these words, case-insensitive: {KW_TEXT}. Why: the queues also hold off-topic tickets (for example software questions), and the tuning and test emails should look like Spreekorb shop email.
- Rule 1, start of word (prefix match): a keyword must match at the start of a word, so "orders" and "returned" match, but "border" does not. Caveat: "charge" also matches "charger" and "charging", so a few off-topic tickets can pass.
- Rule 2, idioms ignored: before matching, the idioms {IDIOM_TEXT} are removed (case-insensitive, whole words, any whitespace between the words) from the text used for matching only. The stored subject and body are not changed. Why: these idioms contain a keyword but are not about shop topics.
- Tickets left after the filter: {len(tix)}.
- Stop rule: if fewer than 200 tickets remain after the filter, the notebook prints a WARNING and stops before sampling and writing. The sample would then be topped up with synthetic emails (not done in this notebook). It also stops if any queue has fewer than 10 tickets.
- Stratified sample (random_state 42): the test set ({n_test}) is 10 random tickets per queue (4 queues = 40) plus 10 random tickets from the remaining pool. The tuning set ({n_tune}) is a random draw from what is left, so tuning and test never overlap. Added `ticket_id` (tuning T001 to T150, test T151 to T200), `split` (tuning or test) and `timing_test` (True for {n_timing} test rows, random).
- Emails and phone numbers in subject, body and answer are replaced by [EMAIL] and [PHONE]. The phone pattern can also mask long number strings such as dates.

Held-out test set, tickets per queue and language (computed when the file was written):

{TEST_TABLE}

## What is synthetic
- `synthetic_team_pulse.csv` and its summary are fully made up (synthetic = True). The pulse is illustrative, not Spreekorb staff. Team labels (Pulse team A to H) are not linked to Shopzilla teams. The answer weights are placeholder assumptions, not data.
- The Tobi-Bueck tickets are synthetic (stated on the Hugging Face dataset card).
- Shopzilla is a public dataset for a different company; Spreekorb is invented. Treat results as illustration.

## Pseudonymisation, minimum group and the Other bucket
No agent, supervisor or manager names are stored. Teams are shown only as Team NN. Minimum group size is {MIN_GROUP}: a team-month needs at least {MIN_GROUP} distinct agents, otherwise its rows are shown as Other. If Other has fewer than {MIN_GROUP} distinct agents in a month, those rows are dropped. Rows dropped: {dropped_all} (all channels) and {dropped_email} (email). Other share of rows: {other_all:.3f} (all channels) and {other_email:.3f} (email). The pulse is team level only, with at least {MIN_GROUP} answers per team and no respondent id.

Pseudonymised, not anonymous: timestamps and free text could still allow re-identification together with public data. Customer names in free text are not fully masked, so review before any OpenAI call.

## Test set rule
`tobi_test_50.csv` is read-only. Never use it for prompt tuning. Use it only for the final evaluation. timing_test rows measure latency only, never prompt tuning.

`langsmith/heldout_priority_labels.csv` ({n_test} rows, made by notebook R1-02 from tobi_test_50.csv) is the labelling sheet. `priority_human`, `queue_human` and `note` start blank. The notebook keeps an existing sheet only if it has the same ticket ids and labels already filled; otherwise the old sheet is copied to the backup folder and a new blank one is written.

## Open items
- Shopzilla licence: to confirm.

## Compliance (before real use)
- Team SLA and CSAT views need works council (Betriebsrat) agreement (BetrVG par. 87(1) no. 6) and a data protection impact assessment (DPIA).
- The EU AI Act assessment for worker monitoring is open.
- Sending tickets to OpenAI needs a data processing agreement (DPA) and a transfer check.
'''
OUT['readme'].write_text(readme, encoding='utf-8')
print('README written:', OUT['readme'])

In [ ]:
# 3e Create: langsmith/heldout_priority_labels.csv (blank labelling sheet for the 50 held-out emails)
# R1-02 writes this sheet; R1-05 only reads it.
LABEL_COLS = ['email_id', 'subject', 'body', 'tobi_queue', 'tobi_priority', 'priority_human', 'note', 'queue_human']
new_ids = set(sample[sample['split'] == 'test']['ticket_id'])

def read_sheet(p):
    # returns (ids in the sheet, True if a label is already typed in)
    if not p.exists():
        return set(), False
    old_sheet = pd.read_csv(p, dtype=str).fillna('')
    ids = set(old_sheet['email_id']) if 'email_id' in old_sheet else set()
    filled = any(c in old_sheet and old_sheet[c].str.strip().ne('').any() for c in ['priority_human', 'queue_human'])
    return ids, filled

old_ids, old_filled = read_sheet(OUT['labels'])
# keep the old sheet only if it is for the same 50 emails AND has labels
labels_kept = old_ids == new_ids and old_filled
if labels_kept:
    print('BRANCH keep: same 50 ids and labels already filled, sheet NOT overwritten:', OUT['labels'].relative_to(REPO))
else:
    if OUT['labels'].exists():
        BK.mkdir(parents=True, exist_ok=True)
        shutil.copy2(OUT['labels'], BK / 'heldout_priority_labels_old.csv')
        print('Old sheet copied to', (BK / 'heldout_priority_labels_old.csv').relative_to(REPO), '| old labels filled:', old_filled, '| same ids:', old_ids == new_ids)
    test_rows = sample[sample['split'] == 'test']
    lab = pd.DataFrame({
        'email_id': test_rows['ticket_id'],
        'subject': test_rows['subject'].fillna('').astype(str).str.strip(),
        'body': test_rows['body'].fillna('').astype(str).str.strip(),
        'tobi_queue': test_rows['queue'],
        'tobi_priority': test_rows['priority'],
        'priority_human': '', 'note': '', 'queue_human': '',
    })
    OUT['labels'].parent.mkdir(parents=True, exist_ok=True)
    lab[LABEL_COLS].to_csv(OUT['labels'], index=False)
    print('BRANCH write: wrote', OUT['labels'].relative_to(REPO), len(lab), 'rows, labels blank.')

In [ ]:
# 4 Verify: one PASS/FAIL per acceptance item (A = spec item, X = extra check)
c = pd.read_csv(OUT['contacts'])
e = pd.read_csv(OUT['email'])
tu = pd.read_csv(OUT['tuning'])
te = pd.read_csv(OUT['test'])
pu = pd.read_csv(OUT['pulse'])
ps = pd.read_csv(OUT['pulse_summary'])
n_before = len(results)   # only count the checks of this cell in the overall result

# A1 Shopzilla
check('A1 email file has only Email rows', len(e) > 0 and set(e['channel_name']) == {'Email'}, str(len(e)) + ' rows')
check('A1 contacts file keeps all 3 channels', set(c['channel_name']) == {'Inbound', 'Outcall', 'Email'}, str(len(c)) + ' rows')
banned = ['Agent_name', 'Supervisor', 'Manager', 'Order_id', 'connected_handling_time', 'Unique id', 'Customer_City', 'Agent Shift', 'Tenure Bucket']
check('A1 banned columns are gone', not any(b in df.columns for df in (c, e) for b in banned))
check('A1 teams look like Team NN or Other', all(t == 'Other' or t.startswith('Team ') for df in (c, e) for t in df['team'].unique()))

# response_hours: no negatives, response_valid matches
for name, df in [('contacts', c), ('email', e)]:
    ok = (df['response_hours'].dropna() >= 0).all() and (df['response_valid'] == df['response_hours'].notna()).all()
    check('X response_hours >= 0 and response_valid consistent (' + name + ')', ok, str(int((~df['response_valid']).sum())) + ' invalid rows')

# min group flag (the 5-distinct-agents rule itself is asserted in 3a before Agent_name is dropped)
for name, df in [('contacts', c), ('email', e)]:
    check('X saved group_ok flag all True (rule enforced in 3a) (' + name + ')', bool(df['group_ok'].all()), 'Other share ' + str(round((df['team'] == 'Other').mean(), 3)))

# names: whole-word scan of Shopzilla outputs, pulse and README only (not Tobi text); tokens are never printed
rd = OUT['readme'].read_text(encoding='utf-8')
for label, text in [('contacts', OUT['contacts'].read_text(encoding='utf-8')), ('email', OUT['email'].read_text(encoding='utf-8')),
                    ('pulse', OUT['pulse'].read_text(encoding='utf-8')), ('pulse summary', OUT['pulse_summary'].read_text(encoding='utf-8')), ('README', rd)]:
    hits = NAME_TOK & set(re.findall(r'[^\W\d_]+', text.lower()))
    check('X no agent/supervisor/manager name as a whole word (' + label + ')', len(hits) == 0, str(len(hits)) + ' distinct tokens hit')
del NAME_TOK   # token set is gone
rem = pd.concat([c['Customer Remarks'], e['Customer Remarks']]).dropna()
check('X no email or phone left in Customer Remarks', rem.str.count(EMAIL_RE).sum() + rem.str.count(PHONE_RE).sum() == 0)

# A2 Tobi-Bueck
both = pd.concat([tu, te])
check('A2 all 4 e-commerce queues present after the keyword filter and in the test set, none other', set(tix['queue']) == set(QUEUES) and set(te['queue']) == set(QUEUES) and set(both['queue']) <= set(QUEUES))
check('A2 both en and de present, none other', set(filtered['language']) == {'en', 'de'} and set(both['language']) <= {'en', 'de'})
check('A2 latest version only (400)', LATEST == 400 and set(both['version'].astype(float)) == {400.0})
check('A2 no duplicate subject+body in sample', not both.duplicated(['subject', 'body']).any())
check('X Tobi sample has 0 emails/phones after masking', pii_count(both) == 0)
check('A2 e-commerce filter applied: every sampled row matches the word-start keyword pattern (idioms ignored)', bool(has_keyword(both).all()), str(int(has_keyword(both).sum())) + ' of ' + str(len(both)) + ' rows match')
check('A2 filter lists: 24 keywords, 3 idioms', len(SHOP_KEYWORDS) == 24 and len(IDIOMS) == 3)
print('INFO tickets remaining after the keyword filter:', len(tix), '(stop rule is < 200)')

# self-test of the keyword rule (prefix rule: 'charger' counts, see README caveat)
def kw(text):
    return bool(has_keyword(pd.DataFrame({'subject': [''], 'body': [text]})).iloc[0])
tests = {'border': False, 'orders': True, 'in order to help': False, 'in order to get a refund': True,
         'charger cable': True, 'in  order to': False}
bad = [t for t, want in tests.items() if kw(t) != want]
check('A2 keyword rule self-test (6 cases)', not bad, 'failed: ' + str(bad) if bad else 'border F, orders T, idiom F, idiom+refund T, charger T (prefix), double space F')

# A3 Synthetic pulse
check('A3 pulse has 80 rows and the right columns', len(pu) == 80 and list(pu.columns) == ['team', 'uses_ai_answer', 'why_category', 'free_text', 'synthetic'])
check('A3 pulse labelled synthetic', bool(pu['synthetic'].all()) and bool(ps['synthetic'].all()))
check('A3 pulse answers Yes/No and valid reasons', set(pu['uses_ai_answer']) <= {'Yes', 'No'} and set(pu['why_category']) <= set(texts))
check('A3 pulse team level: every team >= 5 answers', pu['team'].value_counts().min() >= MIN_GROUP, str(pu['team'].nunique()) + ' teams')
check('A3 pulse teams are neutral labels, not Shopzilla teams', all(t.startswith('Pulse team ') for t in pu['team'].unique()))
check('A3 pulse has no id or name column', not any(col.lower() == 'id' or 'name' in col.lower() for col in pu.columns))
check('A3 pulse summary shows no count under 5', all(v == '<5' or int(v) >= MIN_GROUP for v in ps['count'].astype(str)))

# A4 Held-out test kept apart
check('A4 sizes are 150 and 50', len(tu) == 150 and len(te) == 50, str(len(tu)) + ' / ' + str(len(te)))
check('A4 no overlap (ticket_id, subject+body)',
      not set(tu['ticket_id']) & set(te['ticket_id']) and not set(zip(tu['subject'], tu['body'])) & set(zip(te['subject'], te['body'])))
check('A4 test set has >= 10 tickets per queue (stratified)', te['queue'].value_counts().reindex(QUEUES).fillna(0).min() >= 10, str(te['queue'].value_counts().to_dict()))
check('A4 split labels correct', set(tu['split']) == {'tuning'} and set(te['split']) == {'test'})
check('A4 ticket ids: tuning T001-T150, test T151-T200', list(tu['ticket_id']) == ['T' + str(i).zfill(3) for i in range(1, 151)] and sorted(te['ticket_id']) == ['T' + str(i).zfill(3) for i in range(151, 201)])
check('A4 10 timing flags, all in test', te['timing_test'].sum() == 10 and tu['timing_test'].sum() == 0)
# columns must be the same as before: raw columns in raw order, plus ticket_id first and split, timing_test last
expected_cols = ['ticket_id'] + list(tix_raw.columns) + ['split', 'timing_test']
old_test = BK / OUT['test'].relative_to(REPO)
old_cols = list(pd.read_csv(old_test, nrows=0).columns) if old_test.exists() else expected_cols
check('A4 columns unchanged (tuning and test)', list(tu.columns) == expected_cols == old_cols and list(te.columns) == expected_cols, str(len(expected_cols)) + ' columns')

# A4b held-out labelling sheet in langsmith/
hl = pd.read_csv(OUT['labels'], dtype=str).fillna('')
check('A4b heldout sheet: 7 old columns + queue_human', list(hl.columns) == LABEL_COLS, str(list(hl.columns)))
check('A4b heldout sheet: 50 rows, ids match tobi_test_50', len(hl) == 50 and sorted(hl['email_id']) == sorted(te['ticket_id']))
blank_ok = labels_kept or (hl['queue_human'].eq('').all() and hl['priority_human'].eq('').all())
check('A4b heldout sheet: queue_human (and priority_human) blank, or your labels kept', bool(blank_ok), 'kept your labels' if labels_kept else 'blank')

# A5 Raw not committed, links in README
tracked = subprocess.run(['git', 'ls-files', 'data/raw'], cwd=REPO, capture_output=True, text=True).stdout.strip()
check('A5 raw files not tracked by git', tracked == '', tracked or 'none tracked')
check('A5 data/raw in .gitignore', 'data/raw/' in (REPO / '.gitignore').read_text())
check('A5 README has Kaggle link', 'https://www.kaggle.com/datasets/ddosad/ecommerce-customer-service-satisfaction' in rd)
check('A5 README has Hugging Face link', 'https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets' in rd)
check('X README says pseudonymised, placeholder weights, test rule, compliance', all(s in rd for s in ['pseudonymised', 'placeholder assumptions', 'read-only', 'Betriebsrat', 'DPIA']))
check('X README mentions the e-commerce filter, keywords, start-of-word rule, idioms, stop rule, stratified rule and charger caveat',
      'E-commerce keyword filter' in rd and all(k in rd for k in SHOP_KEYWORDS) and 'start of a word' in rd
      and all(i in rd for i in IDIOMS) and 'Stop rule' in rd and 'fewer than 200' in rd
      and 'Stratified sample' in rd and 'charger' in rd)
check('X README says Tobi-Bueck is synthetic and has no open item about it', 'tickets are synthetic (stated on the Hugging Face dataset card)' in rd and 'Whether the Tobi-Bueck tickets are synthetic' not in rd)

mine = results[n_before:]
fails = [r for r in mine if r[1] == 'FAIL']
print('\nVERIFY SUMMARY:', len(mine) - len(fails), 'PASS,', len(fails), 'FAIL')
print('OVERALL:', 'PASS' if not fails else 'FAIL')

In [ ]:
# 5 Unchanged: raw files, AGENTS.md, CLAUDE.md, specs, .gitignore, other notebooks, other langsmith files and the policy cases keep the same hash
AFTER = snapshot()
check('hashed some files before', len(BEFORE) > 0, str(len(BEFORE)) + ' files')
changed = sorted(k for k in BEFORE if AFTER.get(k) != BEFORE[k])   # missing file counts as changed
check('raw files unchanged', not any(k.startswith('data/raw') for k in changed))
check('R1-05 notebook, other langsmith files and policy cases unchanged', not any(k.startswith(('langsmith', 'notebooks')) or k.endswith('synthetic_policy_cases.csv') for k in changed))
check('no other existing file changed', changed == [], str(changed) if changed else 'ok, ' + str(len(AFTER)) + ' files compared')

In [ ]:
# 6 Undo (only if RUN_UNDO = True): restore backups (incl. README and the heldout sheet), delete files that did not exist before
if RUN_UNDO:
    for key, p in OUT.items():
        if existed[key]:
            shutil.copy2(BK / p.relative_to(REPO), p)
        else:
            p.unlink(missing_ok=True)
    check('undo', all(p.exists() == existed[k] for k, p in OUT.items()), 'outputs restored or removed')
else:
    print('Undo skipped (RUN_UNDO = False)')

In [ ]:
# 7 Git: print only, never run. Ayhan commits.
print('git status')
print('git add data/processed data/README.md langsmith/heldout_priority_labels.csv notebooks/R1-02_data_prep.ipynb')
print('git commit -m "R1-02 fix: e-commerce keyword filter, stratified sample, new held-out set and labelling sheet"')
print('git push')